# Article 1 v3 — reparto entre clientes y dentro de cada cliente

Diagnóstico de **índices reales guardados**; no entrena ni modifica particiones.
Funciones de cada conjunto y política de redondeo: [README](../README.md).
Ejecutar primero la etapa `partition` para el dataset/seed/regímenes elegidos.

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "article1" / "partitioning.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from article1 import REGIMES
from article1.partitioning import ROLES, SPLIT_FRACTIONS, load_partitions

DATASET = "mnist"  # mnist, fmnist, cifar
SEED = 42          # repetir también con 43 y 44
SELECTED_REGIME = "alpha0p1"
OUT = ROOT / "OUTPUTS" / "article1_v3"
DATA_DIR = ROOT / "data"
REGIMES_TO_SHOW = list(REGIMES)
SAVE_FIGURES = True

## Leer las particiones y comprobar los invariantes

Se cargan únicamente las etiquetas del entrenamiento oficial. La lectura verifica protocolo, huellas de índices, etiquetas, disjunción y cobertura exhaustiva. Un conjunto local vacío detiene el análisis. No se consultan resultados de modelos ni etiquetas del test oficial.

In [ ]:
from torchvision import datasets
classes = {"mnist": datasets.MNIST, "fmnist": datasets.FashionMNIST, "cifar": datasets.CIFAR10}
public_train = classes[DATASET](str(DATA_DIR), train=True, download=True)
labels = np.asarray(public_train.targets, dtype=np.int64)
partitions, counts, summaries = {}, {}, []
master_proxy = None
for regime in REGIMES_TO_SHOW:
    path = OUT / "partitions" / f"{DATASET}-seed{SEED}-{regime}"
    if not (path / "metadata.json").is_file():
        raise FileNotFoundError(f"Falta {path}. Ejecuta primero la etapa partition; no se generan datos ficticios.")
    proxy, clients, metadata = load_partitions(path, labels)
    assert (metadata["dataset"], metadata["seed"], metadata["regime"]) == (DATASET, SEED, regime)
    if master_proxy is None:
        master_proxy = proxy
    else:
        np.testing.assert_array_equal(master_proxy, proxy)
    partitions[regime] = clients
    counts[regime] = {role: np.array([np.bincount(labels[c[role + "_idx"]], minlength=10) for c in clients]) for role in ROLES}
    counts[regime]["private"] = sum(counts[regime][role] for role in ROLES)
    for cid in range(10):
        item = {"regime": regime, "client": cid, "private": int(counts[regime]["private"][cid].sum())}
        for role in ROLES:
            item[role] = int(counts[regime][role][cid].sum())
            item[role + "_fraction"] = item[role] / item["private"]
        item["train_classes_without_expertise"] = int(((counts[regime]["train"][cid] > 0) & (counts[regime]["expertise"][cid] == 0)).sum())
        summaries.append(item)
summary = pd.DataFrame(summaries)
display(pd.DataFrame({"class": range(10), "proxy_count": np.bincount(labels[master_proxy], minlength=10)}))
display(summary)
print(f"Verificado: {len(labels)} ejemplos oficiales de train, {len(master_proxy)} públicos y {len(labels)-len(master_proxy)} privados. Protocolo v3.")

In [ ]:
def heatmap(ax, values, title, norm, *, percent=False, annotate=False):
    cmap = plt.get_cmap("viridis").copy()
    cmap.set_bad("#dddddd")
    im = ax.imshow(np.ma.masked_invalid(values), cmap=cmap, norm=norm, aspect="auto")
    ax.set(title=title, xlabel="Clase", ylabel="Cliente", xticks=range(10), yticks=range(10))
    if annotate:
        for k in range(10):
            for c in range(10):
                value = values[k, c]
                if np.isfinite(value):
                    text = f"{100*value:.0f}" if percent else str(int(value))
                    ax.text(c, k, text, ha="center", va="center", fontsize=7,
                            color="black" if norm(value) > .6 else "white")
    return im

figures = {}
regime_labels = {"iid": "IID", "alpha1p0": "α=1.0", "alpha0p5": "α=0.5", "alpha0p1": "α=0.1", "multi": "Multi", "single": "Single"}

## Entre clientes: recuentos absolutos y composición

La escala compartida de recuentos permite ver diferencias de tamaño. La segunda figura normaliza por cliente y muestra composición de clases; no debe utilizarse para comparar volúmenes absolutos. El orden de regímenes es categórico.

In [ ]:
for normalized in (False, True):
    ncols = 3
    nrows = (len(REGIMES_TO_SHOW) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4*nrows), squeeze=False, layout="constrained")
    vmax = 1 if normalized else max(counts[r]["private"].max() for r in REGIMES_TO_SHOW)
    for ax, regime in zip(axes.flat, REGIMES_TO_SHOW):
        values = counts[regime]["private"].astype(float)
        if normalized:
            values = values / values.sum(axis=1, keepdims=True)
        im = heatmap(ax, values, regime_labels[regime], Normalize(0, vmax))
    for ax in list(axes.flat)[len(REGIMES_TO_SHOW):]:
        ax.set_visible(False)
    fig.colorbar(im, ax=list(axes.flat), label="Fracción de los datos del cliente" if normalized else "Número de ejemplos")
    fig.suptitle(f"{DATASET} · seed {SEED} · distribución privada entre clientes")
    figures["inter_composition" if normalized else "inter_counts"] = fig
    plt.show()

## Dentro de cada cliente: train, validation y expertise

Elige `SELECTED_REGIME` arriba y repite para los demás. La escala de recuentos es común. La figura de fracciones divide cada celda por el total privado de esa pareja cliente–clase: los valores de referencia son 70%, 10% y 20%. Gris significa que la clase no fue asignada al cliente; cero significa que sí fue asignada pero no aparece en ese split.

In [ ]:
assert SELECTED_REGIME in counts
selected = counts[SELECTED_REGIME]
fig, axes = plt.subplots(1, 3, figsize=(16, 5), layout="constrained")
norm = Normalize(0, max(selected[role].max() for role in ROLES))
for ax, role in zip(axes, ROLES):
    im = heatmap(ax, selected[role].astype(float), role, norm, annotate=True)
fig.colorbar(im, ax=list(axes), label="Número de ejemplos")
fig.suptitle(f"{DATASET} · seed {SEED} · {regime_labels[SELECTED_REGIME]} · reparto local")
figures[f"intra_counts_{SELECTED_REGIME}"] = fig
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(16, 5), layout="constrained")
for ax, role in zip(axes, ROLES):
    fractions = np.divide(selected[role], selected["private"], out=np.full((10, 10), np.nan), where=selected["private"] > 0)
    im = heatmap(ax, fractions, f"{role} · objetivo {100*SPLIT_FRACTIONS[role]:.0f}%", Normalize(0, 1), percent=True, annotate=True)
fig.colorbar(im, ax=list(axes), label="Fracción del total cliente–clase (anotaciones en %)")
fig.suptitle(f"{DATASET} · seed {SEED} · {regime_labels[SELECTED_REGIME]} · redondeo por clase")
figures[f"intra_fractions_{SELECTED_REGIME}"] = fig
plt.show()

## Soporte escaso y exportación

Los umbrales de recuentos siguientes son descripciones, no reglas nuevas para M. Tener pocas observaciones afecta a precisión, no implica leakage. Revisa especialmente clases presentes en train sin observaciones en expertise.

In [ ]:
support_rows = []
for regime in REGIMES_TO_SHOW:
    for cid in range(10):
        for c in range(10):
            n = int(counts[regime]["private"][cid, c])
            if n:
                support_rows.append({"regime": regime, "client": cid, "class": c,
                                     **{role: int(counts[regime][role][cid, c]) for role in ROLES}})
support = pd.DataFrame(support_rows)
display(support[support.expertise.lt(5)])
display(summary.groupby("regime", sort=False)[["private", *ROLES]].agg(["min", "max"]))
if SAVE_FIGURES:
    destination = OUT / "figures" / "partitions" / f"{DATASET}-seed{SEED}"
    destination.mkdir(parents=True, exist_ok=True)
    for name, fig in figures.items():
        for extension in ("png", "pdf"):
            fig.savefig(destination / f"{name}.{extension}", dpi=180, bbox_inches="tight")
    summary.to_csv(destination / "client_split_sizes.csv", index=False)
    support.to_csv(destination / "client_class_counts.csv", index=False)
    print(destination)